# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [10]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:



In [8]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.



In [8]:
print("--- Movies ---")
display(movies[movies["movieId"] == 1])

print("--- Ratings ---")
display(ratings[ratings["movieId"] == 1])

print("--- Links ---")
display(links[links["movieId"] == 1])

print("--- Credits ---")
display(credits[credits["tmdbId"] == 862])

print("--- Keywords ---")
display(keywords[keywords["tmdbId"] == 862])

display(movies.sort_values(by="revenue", ascending=False).head(10))

display(movies.sort_values(by="release_date", ascending=False).head(10))

movies[movies.isna().any(axis=1)]

display(movies[movies.isna().any(axis=1) & (movies["revenue"] != 0)])

# Group by movieId to get the average, median, and total rating count
rating_summary = ratings.groupby("movieId").agg(
    rating_avg=("rating", "mean"),
    rating_median=("rating", "median"),
    rating_count=("userId", "count")
).reset_index()

print("--- Merge Rating Summary to Movie df ---")
rating_summary_with_titles = rating_summary.merge(
    movies[["movieId", "title", "vote_average", "vote_count"]], 
    on="movieId", 
    how="left"
)

# Display top 100 with titles
display(rating_summary_with_titles.sort_values(by="rating_count", ascending=False).head(100))

--- Movies ---


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995


--- Ratings ---


,userId,movieId,rating,timestamp,rating_datetime
494,7,1,3.0,851866703,1996-12-29 13:38:23
697,9,1,4.0,938629179,1999-09-29 18:19:39
887,13,1,5.0,1331380058,2012-03-10 11:47:38
959,15,1,2.0,997938310,2001-08-16 05:05:10
3099,19,1,3.0,855190091,1997-02-06 00:48:11
...,...,...,...,...,...
98341,660,1,2.5,1436680062,2015-07-12 05:47:42
98523,663,1,4.0,1438397999,2015-08-01 02:59:59
98549,664,1,3.5,1362421730,2013-03-04 18:28:50
99664,670,1,4.0,938782344,1999-10-01 12:52:24


--- Links ---


...

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
7390,72998,19995,tt0499549,Avatar,Avatar,en,2009-12-10,162.0,237000000,2.787965e+09,185.070892,7.2,12114.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Ingenious Film Partners', 'id': 289...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Fantasy|Science Fiction,Action,2009
8746,122886,140607,tt2488496,Star Wars: The Force Awakens,Star Wars: The Force Awakens,en,2015-12-15,136.0,245000000,2.068224e+09,31.626013,7.5,7993.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Lucasfilm', 'id': 1}, {'name': 'Tru...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Science Fiction|Fantasy,Action,2015
1355,1721,597,tt0120338,Titanic,Titanic,en,1997-11-18,194.0,200000000,1.845034e+09,26.889070,7.5,7770.0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Drama|Romance|Thriller,Drama,1997
7864,89745,24428,tt0848228,The Avengers,The Avengers,en,2012-04-25,143.0,220000000,1.519558e+09,89.887648,7.4,12000.0,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Science Fiction|Action|Adventure,Science Fiction,2012
8700,117529,135397,tt0369610,Jurassic World,Jurassic World,en,2015-06-09,124.0,150000000,1.513529e+09,32.790475,6.5,8842.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Universal Studios', 'id': 13}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Science Fiction|Thriller,Action,2015
8820,130634,168259,tt2820852,Furious 7,Furious 7,en,2015-04-01,137.0,190000000,1.506249e+09,27.275687,7.3,4253.0,"[{'id': 28, 'name': 'Action'}]","[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'JP', 'name': 'Japan'}, {'iso_...",False,Released,Action,Action,2015
8749,122892,99861,tt2395427,Avengers: Age of Ultron,Avengers: Age of Ultron,en,2015-04-22,141.0,280000000,1.405404e+09,37.379420,7.3,6908.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Marvel Studios', 'id': 420}, {'name...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Science Fiction,Action,2015
7816,88125,12445,tt1201607,Harry Potter and the Deathly Hallows: Part 2,Harry Potter and the Deathly Hallows: Part 2,en,2011-07-07,130.0,125000000,1.342000e+09,24.990737,7.9,6141.0,"[{'id': 10751, 'name': 'Family'}, {'id': 14, '...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Family|Fantasy|Adventure,Family,2011
8429,106696,109445,tt2294629,Frozen,Frozen,en,2013-11-27,102.0,150000000,1.274219e+09,24.248243,7.3,5440.0,"[{'id': 16, 'name': 'Animation'}, {'id': 12, '...","[{'name': 'Walt Disney Pictures', 'id': 2}, {'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Adventure|Family,Animation,2013
8280,102125,68721,tt1300854,Iron Man 3,Iron Man 3,en,2013-04-18,130.0,200000000,1.215440e+09,23.721243,6.8,8951.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Marvel Studios', 'id': 420}]","[{'iso_3166_1': 'CN', 'name': 'China'}, {'iso_...",False,Released,Action|Adventure|Science Fiction,Action,2013


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
9023,151307,373355,tt5278868,The Lovers and the Despot,The Lovers and the Despot,en,2016-09-23,100.0,0,0.0,0.744523,7.0,4.0,"[{'id': 99, 'name': 'Documentary'}]",[],[],False,Released,Documentary,Documentary,2016
9081,163949,391698,tt2531318,The Beatles: Eight Days a Week - The Touring Y...,The Beatles: Eight Days a Week - The Touring Y...,en,2016-09-15,99.0,0,0.0,7.078301,7.6,92.0,"[{'id': 99, 'name': 'Documentary'}, {'id': 104...","[{'name': 'Imagine Entertainment', 'id': 23}, ...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Documentary|Music,Documentary,2016
8747,122888,271969,tt2638144,Ben-Hur,Ben-Hur,en,2016-08-17,125.0,100000000,94061311.0,11.510210,5.3,642.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Drama|Action,Adventure,2016
9073,161582,338766,tt2582782,Hell or High Water,Hell or High Water,en,2016-08-12,102.0,12000000,37589296.0,12.565896,7.2,1304.0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'name': 'Sidney Kimmel Entertainment', 'id':...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Crime|Drama|Thriller|Western,Crime,2016
9078,162542,392572,tt5165344,Rustom,रुस्तम,hi,2016-08-12,150.0,1000000,0.0,7.333139,7.3,25.0,"[{'id': 53, 'name': 'Thriller'}, {'id': 10749,...","[{'name': 'KriArj Entertainment', 'id': 91689}]","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Thriller|Romance,Thriller,2016
9079,162672,402672,tt3859980,Mohenjo Daro,Mohenjo Daro,hi,2016-08-11,155.0,15050000,16180000.0,1.423358,6.7,26.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'UTV Motion Pictures', 'id': 2320}, ...","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Adventure|Drama|History|Romance,Adventure,2016
8885,135536,297761,tt1386697,Suicide Squad,Suicide Squad,en,2016-08-02,123.0,175000000,745600054.0,42.965027,5.9,7717.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'DC Comics', 'id': 429}, {'name': 'D...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Crime|Fantasy|Science Fiction,Action,2016
9076,161918,390989,tt4831420,Sharknado 4: The 4th Awakens,Sharknado 4: The 4th Awakens,en,2016-07-31,85.0,0,0.0,4.574494,4.3,88.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[{'name': 'The Asylum', 'id': 1311}, {'name': ...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Horror|Science Fiction,Comedy,2016
9080,163056,315011,tt4262980,Shin Godzilla,シン・ゴジラ,ja,2016-07-29,120.0,15000000,77000000.0,9.285519,6.6,152.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Cine Bazar', 'id': 5896}, {'name': ...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",False,Released,Action|Adventure|Drama|Horror|Science Fiction,Action,2016
9061,160438,324668,tt4196776,Jason Bourne,Jason Bourne,en,2016-07-27,123.0,120000000,415484914.0,19.133256,5.9,2386.0,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...","[{'name': 'The Kennedy/Marshall Company', 'id'...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Action|Thriller,Action,2016


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
4544,6269,51927,tt0334416,Stevie,Stevie,en,2002-09-09,140.0,0,97000.0,0.489997,6.7,13.0,"[{'id': 99, 'name': 'Documentary'}, {'id': 18,...",[],[],False,NaN,Documentary|Drama|Foreign,Documentary,2002
5470,8622,1777,tt0361596,Fahrenheit 9/11,Fahrenheit 9/11,en,2004-06-25,122.0,6000000,119114517.0,6.839460,6.9,403.0,[],"[{'name': 'BIM Distribuzione', 'id': 225}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,NaN,NaN,2004


--- Merge Rating Summary to Movie df ---


,movieId,rating_avg,rating_median,rating_count,title,vote_average,vote_count
321,356,4.054252,4.0,341,Forrest Gump,8.2,8147.0
266,296,4.256173,4.5,324,Pulp Fiction,8.3,8670.0
284,318,4.487138,5.0,311,The Shawshank Redemption,8.5,8358.0
525,593,4.138158,4.0,304,The Silence of the Lambs,8.1,4549.0
232,260,4.221649,4.5,291,Star Wars,8.1,6778.0
...,...,...,...,...,...,...,...
6872,58559,4.235537,4.5,121,The Dark Knight,8.3,12269.0
1902,2396,3.966942,4.0,121,Shakespeare in Love,6.8,831.0
958,1206,4.000000,4.0,121,A Clockwork Orange,8.0,3432.0
37,39,3.550000,3.5,120,Clueless,6.9,828.0


In [27]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


...

,dtype
tmdbId,int64
director,str
top_cast,str


,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [ ]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [6]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [7]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


Preliminary ABT shape: (9082, 32)


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,high_user_rating,well_rated_enough_data
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,1,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,0,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,0,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,0,0
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,0,0


,missing_rate
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
first_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276
primary_genre,0.003854
genres_list,0.003854
director,0.002643


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [8]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [9]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
